In [2]:
import cv2
import mediapipe as mp
import numpy as np
import os
import glob
import csv
from tqdm import tqdm

# ================= 配置区域 =================
# 阈值设置 (需要根据实际情况微调)
EYE_ROLL_THRESHOLD = 0.30    # 虹膜距离上眼睑的比例 (越小越靠上，<0.30 判定为翻白眼)
SIDE_EYE_LEFT_THR = 0.30     # 虹膜水平比例 < 0.30 判定为向左看
SIDE_EYE_RIGHT_THR = 0.70    # 虹膜水平比例 > 0.70 判定为向右看
BLINK_THRESHOLD = 0.04       # 眼睛纵横比 (EAR) 小于此值判定为闭眼，不进行视线检测
MIN_FACE_SIZE = 40           # 脸部框最小宽度(像素)，过滤低分辨率背景杂脸

# 关键点索引 (MediaPipe Face Mesh 标准)
# 左眼
LEFT_IRIS = [468]
LEFT_TOP = [159]
LEFT_BOTTOM = [145]
LEFT_INNER = [133]
LEFT_OUTER = [33]
# 右眼
RIGHT_IRIS = [473]
RIGHT_TOP = [386]
RIGHT_BOTTOM = [374]
RIGHT_INNER = [362]
RIGHT_OUTER = [263]
# ===========================================

try:
    # 尝试访问 solutions
    mp_face_mesh = mp.solutions.face_mesh
    print("MediaPipe solutions 加载成功！")
except AttributeError as e:
    print(f"依然报错: {e}")
    # 打印一下 mp 到底是从哪里导入的
    print(f"MediaPipe 文件路径: {mp.__file__}")

class BatchGazeDetector:
    def __init__(self):
        self.mp_face_mesh = mp.solutions.face_mesh
        self.face_mesh = self.mp_face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=5,  # 允许检测多张脸
            refine_landmarks=True, # 必须开启，用于获取虹膜中心
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

    def _get_distance(self, p1, p2, width, height):
        """计算两点像素距离"""
        return np.sqrt(((p1.x - p2.x) * width) ** 2 + ((p1.y - p2.y) * height) ** 2)

    def _get_ratio(self, point, anchor_min, anchor_max, axis='y'):
        """计算点在两端点之间的相对位置 (0.0 ~ 1.0)"""
        val = point.y if axis == 'y' else point.x
        min_val = anchor_min.y if axis == 'y' else anchor_min.x
        max_val = anchor_max.y if axis == 'y' else anchor_max.x
        
        # 防止分母为0
        total_dist = abs(max_val - min_val)
        if total_dist < 1e-6: return 0.5
        
        # 计算相对位置
        dist_from_min = val - min_val
        # 注意：图像坐标系y轴向下，所以如果是垂直方向，通常上面的点数值小
        if axis == 'y': 
            # 翻转逻辑：我们希望算出“距上眼睑的距离占比”
            # 上眼睑y值小，下眼睑y值大
            # ratio = (current - top) / (bottom - top)
            ratio = (val - min_val) / total_dist
        else:
            # 水平方向：左(小) -> 右(大)
            ratio = (val - min_val) / total_dist
            
        return ratio

    def process_frame(self, frame):
        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.face_mesh.process(rgb_frame)
        
        detections = [] # 存储当前帧所有脸的检测结果

        if results.multi_face_landmarks:
            for face_id, face_landmarks in enumerate(results.multi_face_landmarks):
                landmarks = face_landmarks.landmark
                
                # 1. 过滤过小的人脸 (抗噪)
                face_left_x = landmarks[234].x * w # 脸左耳附近
                face_right_x = landmarks[454].x * w # 脸右耳附近
                face_width = abs(face_right_x - face_left_x)
                if face_width < MIN_FACE_SIZE:
                    continue

                # --- 左眼计算 ---
                l_top = landmarks[LEFT_TOP[0]]
                l_bot = landmarks[LEFT_BOTTOM[0]]
                l_in = landmarks[LEFT_INNER[0]]
                l_out = landmarks[LEFT_OUTER[0]]
                l_iris = landmarks[LEFT_IRIS[0]]
                
                # 眨眼检测 (EAR近似)
                l_h = self._get_distance(l_top, l_bot, w, h)
                l_w = self._get_distance(l_in, l_out, w, h)
                l_ear = l_h / l_w if l_w > 0 else 0
                
                # 视线比率
                # 垂直比率：越小越靠上
                l_v_ratio = self._get_ratio(l_iris, l_top, l_bot, axis='y')
                # 水平比率：越小越靠左(观众视角)
                l_h_ratio = self._get_ratio(l_iris, l_out, l_in, axis='x') # 注意左右眼内外角顺序不同

                # --- 右眼计算 ---
                r_top = landmarks[RIGHT_TOP[0]]
                r_bot = landmarks[RIGHT_BOTTOM[0]]
                r_in = landmarks[RIGHT_INNER[0]]
                r_out = landmarks[RIGHT_OUTER[0]]
                r_iris = landmarks[RIGHT_IRIS[0]]
                
                r_h = self._get_distance(r_top, r_bot, w, h)
                r_w = self._get_distance(r_in, r_out, w, h)
                r_ear = r_h / r_w if r_w > 0 else 0
                
                r_v_ratio = self._get_ratio(r_iris, r_top, r_bot, axis='y')
                # MediaPipe左右眼对称为镜像，需要小心处理水平方向
                # 这里我们统一坐标系：0(画面左) -> 1(画面右)
                r_h_ratio = self._get_ratio(r_iris, r_out, r_in, axis='x') # 使用Outer(右)和Inner(左)做锚点

                # --- 状态判定 ---
                # 1. 优先过滤眨眼/闭眼
                if l_ear < BLINK_THRESHOLD or r_ear < BLINK_THRESHOLD:
                    status = "Blink/Closed"
                else:
                    status = "Normal"
                    
                    # 2. 翻白眼判定 (双眼同时向上)
                    # 垂直比率小说明虹膜靠近上眼睑
                    if l_v_ratio < EYE_ROLL_THRESHOLD and r_v_ratio < EYE_ROLL_THRESHOLD:
                        status = "EYE_ROLL"
                    
                    # 3. 斜视判定 (双眼同时向左 或 双眼同时向右)
                    # 这里简化为单纯的水平移动，不区分内斜视/外斜视
                    # 修正：左眼的out在画面左边，inner在右边。
                    # 我们重新统一逻辑：计算iris在 bbox 里的相对x位置
                    
                    # 重新计算简单的水平归一化坐标
                    l_iris_x = (l_iris.x - l_out.x) / (l_in.x - l_out.x) 
                    r_iris_x = (r_iris.x - r_in.x) / (r_out.x - r_in.x)
                    
                    # 两个眼睛的水平比率：0为画面左侧，1为画面右侧
                    avg_h_ratio = (l_iris_x + r_iris_x) / 2
                    
                    if avg_h_ratio < SIDE_EYE_LEFT_THR:
                        status = "SIDE_EYE_LEFT"
                    elif avg_h_ratio > SIDE_EYE_RIGHT_THR:
                        status = "SIDE_EYE_RIGHT"

                # 记录结果
                bbox = (int(face_left_x), int(landmarks[10].y * h), int(face_width), int(face_width * 1.2))
                detections.append({
                    "face_id": face_id,
                    "status": status,
                    "bbox": bbox,
                    "v_ratio": (l_v_ratio + r_v_ratio) / 2
                })
        
        return detections

def batch_process_videos(input_folder, output_csv):
    detector = BatchGazeDetector()
    video_files = glob.glob(os.path.join(input_folder, "*.mp4")) + \
                  glob.glob(os.path.join(input_folder, "*.avi")) + \
                  glob.glob(os.path.join(input_folder, "*.mkv"))
    
    if not video_files:
        print("未找到视频文件！")
        return

    # 准备CSV输出
    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["Video_Name", "Frame_Index", "Timestamp(s)", "Face_ID", "Gaze_Status", "Roll_Intensity"])
        
        for video_path in tqdm(video_files, desc="Processing Videos"):
            video_name = os.path.basename(video_path)
            cap = cv2.VideoCapture(video_path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            if fps == 0: fps = 25.0
            
            frame_idx = 0
            
            # 为了可视化调试，可以保存一个样本视频 (可选)
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(f'output_{video_name}', fourcc, fps, (int(cap.get(3)), int(cap.get(4))))

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break
                
                detections = detector.process_frame(frame)
                
                for det in detections:
                    # 只有当检测到异常状态（白眼或斜视）时才写入CSV，或者是你想记录每一帧
                    # 这里为了节省空间，只记录非Normal和非Blink的状态，或者你可以全记
                    if det["status"] in ["EYE_ROLL", "SIDE_EYE_LEFT", "SIDE_EYE_RIGHT"]:
                        writer.writerow([
                            video_name, 
                            frame_idx, 
                            round(frame_idx/fps, 3), 
                            det["face_id"], 
                            det["status"],
                            round(det["v_ratio"], 3)
                        ])
                    
                    # 可视化 (如果需要生成demo视频)
                    color = (0, 0, 255) if det["status"] == "EYE_ROLL" else (0, 255, 0)
                    x, y, w, h = det["bbox"]
                    cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
                    cv2.putText(frame, det["status"], (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
                
                # out.write(frame)
                frame_idx += 1
            
            # out.release()
            cap.release()

if __name__ == "__main__":
    # 使用示例
    # 1. 创建 input_videos 文件夹并放入你的视频
    # 2. 运行脚本
    input_dir = "utterances_eyeroll"  # 修改为你的视频文件夹路径
    output_file = "gaze_results.csv" # 结果保存路径
    
    if not os.path.exists(input_dir):
        os.makedirs(input_dir)
        print(f"请将视频放入 {input_dir} 文件夹中")
    else:
        batch_process_videos(input_dir, output_file)
        print(f"处理完成，结果已保存至 {output_file}")

MediaPipe solutions 加载成功！


Processing Videos: 100%|██████████| 1/1 [00:03<00:00,  3.19s/it]

处理完成，结果已保存至 gaze_results.csv
